# Unit 6 — From text to speech

Unit 5 turned speech into text. Unit 6 runs the tape backwards, and the first thing you learn is
that it is not symmetric: **ASR has one right answer and a metric to score it with; TTS has many
right answers and, the course says, no metric at all.**

This notebook is the inline version of `walkthrough.py` — same eight sections, but the plots render
here and the audio is playable.

1. Two models, not one — SpeechT5 writes a spectrogram, HiFi-GAN makes the sound
2. The 81-token front end — a character vocabulary, and the `<unk>` that never warns you
3. Speaker control — 512 floats decide whose voice it is, and who consented
4. Three designs — SpeechT5 vs MMS/VITS vs Bark, and the one-to-many problem
5. TTS data — what the course lists, what still loads, what "good" means
6. Fine-tuning innards — what the TTS collator builds, and why it is Unit 5 inverted
7. How TTS fails — the stop token and text the vocabulary cannot hold
8. Evaluating for real — round-trip WER, a pitch check, and a blind MOS sheet

Everything runs on CPU. A cold machine pulls about 0.86 GB into `~/.cache/huggingface`.

In [ ]:
%matplotlib inline
import numpy as np
import torch
import matplotlib.pyplot as plt
import IPython.display as ipd

MODEL_ID = "microsoft/speecht5_tts"
VOCODER_ID = "microsoft/speecht5_hifigan"
MMS_ID = "facebook/mms-tts-eng"
ASR_ID = "openai/whisper-tiny"
DATASET_ID = "ylacombe/english_dialects"
CONFIG = "northern_female"
XVECTOR_ID = "Matthijs/cmu-arctic-xvectors"

SAMPLING_RATE = 16_000
NUM_MEL_BINS = 80
HOP_SAMPLES = 256          # hop 16 ms x 16 kHz; HiFi-GAN's upsample_rates multiply to 256
REDUCTION_FACTOR = 2
VOICES = ["slt", "clb", "bdl", "rms", "ksp"]
DEMO_TEXT = "the sun provides energy for life on earth"
SEED = 6                   # SpeechT5 applies dropout at INFERENCE, so every call is seeded

torch.set_grad_enabled(False)
print("torch", torch.__version__)

In [ ]:
from transformers import SpeechT5ForTextToSpeech, SpeechT5HifiGan, SpeechT5Processor

processor = SpeechT5Processor.from_pretrained(MODEL_ID)
model = SpeechT5ForTextToSpeech.from_pretrained(MODEL_ID).eval()
vocoder = SpeechT5HifiGan.from_pretrained(VOCODER_ID)

def speak(text, speaker, vocoder=None, seed=SEED):
    ids = processor(text=text, return_tensors="pt")["input_ids"]
    torch.manual_seed(seed)          # not optional: see section 4
    return model.generate_speech(ids, speaker, vocoder=vocoder)

print("reduction_factor:", model.config.reduction_factor,
      "| mel bins:", model.config.num_mel_bins,
      "| speaker dim:", model.config.speaker_embedding_dim)

In [ ]:
from datasets import load_dataset

# The x-vectors live in the "validation" split - it is the only split this dataset publishes.
xvectors = load_dataset(XVECTOR_ID, split="validation")
filenames = xvectors["filename"]

# Match by filename prefix, not a row index: an index would silently repoint if the
# dataset were ever revised.
SPEAKERS = {}
for voice in VOICES:
    idx = next(i for i, f in enumerate(filenames) if f.startswith(f"cmu_us_{voice}_"))
    SPEAKERS[voice] = torch.tensor(xvectors[idx]["xvector"]).unsqueeze(0)

speaker = SPEAKERS["slt"]
print(len(xvectors), "x-vectors of", SPEAKERS["slt"].shape[1], "dims")

## 1 — Text to speech is two models

SpeechT5 does **not** output audio. It outputs a **log-mel spectrogram** — a picture of which
frequencies are loud over time. A separate network, the **vocoder** (HiFi-GAN), turns that picture
into a waveform. You fine-tune the first model; the vocoder stays frozen.

In [ ]:
mel = speak(DEMO_TEXT, speaker)                      # no vocoder -> spectrogram
wav = speak(DEMO_TEXT, speaker, vocoder=vocoder)     # vocoder    -> waveform

print("spectrogram :", tuple(mel.shape), "(frames x mel bins)")
print("waveform    :", tuple(wav.shape))
print("frames x 256:", mel.shape[0] * HOP_SAMPLES, " == samples:", len(wav))

mel_np, wav_np = np.asarray(mel), np.asarray(wav)
secs = mel_np.shape[0] * HOP_SAMPLES / SAMPLING_RATE
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax1.imshow(mel_np.T, aspect="auto", origin="lower", cmap="magma", extent=[0, secs, 0, NUM_MEL_BINS])
ax1.set(ylabel="mel bin", title="SpeechT5 writes this")
ax2.plot(np.arange(len(wav_np)) / SAMPLING_RATE, wav_np, lw=0.4)
ax2.set(xlabel="seconds", ylabel="amplitude", title="HiFi-GAN turns it into this")
ax2.set_xlim(0, secs)
plt.tight_layout(); plt.show()

ipd.Audio(wav_np, rate=SAMPLING_RATE)

The two models meet at one number: a mel frame is **256 samples** of hop (16 ms × 16 kHz), and
HiFi-GAN's `upsample_rates` multiply to 4·4·4·4 = 256. Any vocoder trained on 80-bin mels with the
same hop can be swapped in.

## 2 — The 81-token front end, and the `<unk>` that never warns you

SpeechT5's vocabulary is **81 characters**. Anything outside it becomes `<unk>` *silently* — no
warning, no error, just wrong speech.

The obvious way to audit for that is wrong, and worth watching fail.

In [ ]:
tokenizer = processor.tokenizer
vocab = tokenizer.get_vocab()
unk_id = tokenizer.unk_token_id
print("len(get_vocab())     :", len(vocab), "  (matches config.vocab_size)")
print("tokenizer.vocab_size :", tokenizer.vocab_size, "  (excludes added <mask>, <ctc_blank>)")

# The WRONG audit: compare characters against get_vocab() keys.
print("\nis a space 'missing' from the vocab keys?", " " not in vocab)
print("  -> a sentencepiece model stores the word boundary as U+2581, so a literal ' '")
print("     looks absent. This check rejects every row that contains a space.")

# The RIGHT audit: ask whether the tokenizer emits <unk>.
def unsupported(text):
    return unk_id in tokenizer(text, add_special_tokens=False).input_ids

print("\ndoes the tokenizer actually choke on a space?", unsupported(" "))
print("does it choke on digits?                     ", unsupported("1984"))

In [ ]:
hostile = "In 1984 he paid 25 pounds"
ids = tokenizer(hostile, add_special_tokens=False).input_ids
print(repr(hostile))
print("tokens:", " ".join(tokenizer.convert_ids_to_tokens(ids)))
print("<unk> count:", tokenizer.convert_ids_to_tokens(ids).count("<unk>"))

# The escape hatch the course never mentions: the tokenizer ships a number normalizer.
from transformers import SpeechT5Tokenizer

ntok = SpeechT5Tokenizer.from_pretrained(MODEL_ID, normalize=True)
nids = ntok(hostile, add_special_tokens=False).input_ids
print("\nnormalize=True ->", repr(ntok.decode(nids)))
print("<unk> now      :", ntok.unk_token_id in nids)

So digits *are* recoverable. The hands-on still **drops** those rows, because a training run wants
a predictable corpus more than it wants thirty extra clips.

## 3 — Five hundred and twelve floats decide whose voice it is

SpeechT5 is multi-speaker: alongside the text it takes a 512-dimensional **x-vector** saying *whose*
voice to use.

The course computes these with SpeechBrain. We don't: `speechbrain.pretrained` was renamed to
`speechbrain.inference`, so the course's import is a straight `ImportError`, and SpeechBrain pulls in
`torchaudio`, which this repo deliberately does not install. The pre-computed CMU ARCTIC set does the
same job with no extra dependency.

In [ ]:
voice_waves = {}
for voice in VOICES:
    w = np.asarray(speak(DEMO_TEXT, SPEAKERS[voice], vocoder=vocoder))
    voice_waves[voice] = w
    print(f"cmu_us_{voice:<4} {len(w) / SAMPLING_RATE:.2f}s")

# Same text, same seed, five vectors, five different durations: the x-vector reaches the
# duration mechanism, not just the timbre.
gap = np.zeros(int(0.4 * SAMPLING_RATE), dtype=np.float32)
ipd.Audio(np.concatenate([np.concatenate([voice_waves[v], gap]) for v in VOICES]), rate=SAMPLING_RATE)

In [ ]:
# Magnitude is irrelevant - the decoder pre-net L2-normalises the vector before use.
a = speak(DEMO_TEXT, SPEAKERS["slt"])
b = speak(DEMO_TEXT, SPEAKERS["slt"] * 10.0)
print("scaling the x-vector by 10 changes the output:",
      not (a.shape == b.shape and float((a - b).abs().max()) < 1e-5))
print("-> only the DIRECTION matters, which is why section 8 uses cosine, not distance.")

> **Ethics.** The course's introduction page ends here, and so should this section: a 512-float
> vector is all it takes to put words in someone's voice. Voice data needs explicit, informed consent
> covering purpose, scope and risk. The vectors used here come from CMU ARCTIC, a public research
> corpus recorded for exactly this purpose.

## 4 — Three designs, and the one-to-many problem

| | output | vocoder? | speaker control | languages |
|---|---|---|---|---|
| **SpeechT5** | log-mel | yes (HiFi-GAN) | x-vector | English |
| **Bark** | waveform | no (EnCodec) | voice presets | multilingual |
| **MMS / VITS** | waveform | no (built in) | **none** | 1100+, one checkpoint each |

MMS covers 1100+ languages but ships **one checkpoint per language** and gives you no voice control
at all. Ten languages means ten downloads, each with a fixed voice.

In [ ]:
from transformers import VitsModel, VitsTokenizer

vt = VitsTokenizer.from_pretrained(MMS_ID)
vm = VitsModel.from_pretrained(MMS_ID)
vw = np.asarray(vm(**vt(text=DEMO_TEXT, return_tensors="pt")).waveform).squeeze()
print("MMS/VITS:", round(len(vw) / vm.config.sampling_rate, 2), "s, no vocoder needed")
ipd.Audio(vw, rate=vm.config.sampling_rate)

### The one-to-many problem

The same text has many valid spoken forms. You can watch it happen: two identical calls, same
weights, same x-vector, `model.eval()`, no gradients — and different output.

In [ ]:
ids = processor(text=DEMO_TEXT, return_tensors="pt")["input_ids"]
torch.manual_seed(SEED)
c1 = model.generate_speech(ids, speaker)
c2 = model.generate_speech(ids, speaker)      # no reseed

s1 = speak(DEMO_TEXT, speaker, seed=SEED)
s2 = speak(DEMO_TEXT, speaker, seed=SEED)
print("same seed :", s1.shape[0], "vs", s2.shape[0], "frames, identical =",
      s1.shape == s2.shape and float((s1 - s2).abs().max()) == 0.0)
print("no reseed :", c1.shape[0], "vs", c2.shape[0], "frames  <- different lengths")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, m, label in ((ax1, np.asarray(c1), "run 1"), (ax2, np.asarray(c2), "run 2")):
    ax.imshow(m.T, aspect="auto", origin="lower", cmap="magma")
    ax.set(xlabel="frame", title=f"{label}: {m.shape[0]} frames")
ax1.set(ylabel="mel bin")
plt.suptitle("Identical inputs, different speech (dropout is on at inference)")
plt.tight_layout(); plt.show()

The cause is the speech decoder pre-net, which applies dropout `p=0.5` **even in `eval()`** — the
source comment says so outright, citing Tacotron 2.

**`model.eval()` does not make this model deterministic. `torch.manual_seed()` does.**

## 5 — Choosing TTS data

The course names three challenges: one-to-many (section 4 just proved it), long-distance
dependencies, and the data itself. What makes a corpus *good* for ASR — background noise, varied
channels, spontaneous speech — is exactly what you do not want in TTS.

**Three of the four corpora the course recommends no longer load.** `lj_speech`, `vctk` and
`libritts-r-aligned` are loading-script datasets, and script execution was removed in `datasets` 3.0.
Only `multilingual_librispeech` has been converted to parquet.

In [ ]:
from datasets import Audio

# Streamed, not downloaded: the config is one parquet file of 8 row groups of 100 rows, so
# 40 rows range-reads ~59 MB instead of 395 MB. The source is 48 kHz, so the cast is mandatory.
stream = load_dataset(DATASET_ID, CONFIG, split="train", streaming=True)
stream = stream.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))
clips = [row for _, row in zip(range(40), stream)]

durations = [len(c["audio"]["array"]) / SAMPLING_RATE for c in clips]
lengths = [len(c["text"]) for c in clips]
speakers = {}
for c in clips:
    speakers[c["speaker_id"]] = speakers.get(c["speaker_id"], 0) + 1
print(f"{len(clips)} clips | {min(durations):.1f}-{max(durations):.1f}s | speakers {dict(sorted(speakers.items()))}")

plt.figure(figsize=(6, 4))
plt.scatter(lengths, durations, alpha=0.7)
plt.xlabel("characters of text"); plt.ylabel("seconds of audio")
plt.title("The relationship a TTS model has to learn")
plt.tight_layout(); plt.show()

## 6 — What the TTS collator builds

In Unit 5 the labels were token ids. Here they are a **spectrogram**: one 80-bin vector per frame.
That one difference drives every quirk of the collator.

In [ ]:
batch = []
for c in clips[:4]:
    out = processor(text=c["text"], audio_target=c["audio"]["array"],
                    sampling_rate=SAMPLING_RATE, return_attention_mask=False)
    if not batch:
        print("processor returns labels with shape", np.asarray(out["labels"]).shape)
    out["labels"] = out["labels"][0]          # REQUIRED here - see below
    batch.append(out)

lens = [len(b["labels"]) for b in batch]
print("after [0]                :", np.asarray(batch[0]["labels"]).shape)
print("four clips, label frames :", lens)

padded = processor.pad(
    input_ids=[{"input_ids": b["input_ids"]} for b in batch],
    labels=[{"input_values": b["labels"]} for b in batch],
    return_tensors="pt",
)
labels = padded["labels"].masked_fill(padded.decoder_attention_mask.unsqueeze(-1).ne(1), -100)
target_lengths = torch.tensor(lens)
target_lengths = target_lengths.new([l - l % REDUCTION_FACTOR for l in target_lengths])
print(f"reduction_factor={REDUCTION_FACTOR} rounds:", lens, "->", target_lengths.tolist())
print("padded labels            :", tuple(labels.shape))

> **`out["labels"] = out["labels"][0]` is required here** — the processor wraps the spectrogram in a
> batch dimension of one. This is the exact *inverse* of Unit 5, where the same line was the bug:
> there the labels are a flat list of ids and `[0]` takes a single integer.

The decoder emits 2 mel frames per step, so an odd target length leaves the loss misaligned against
the predictions. And `decoder_attention_mask` is deleted after masking — the model does not take it
during training.

## 7 — How TTS fails

An autoregressive TTS model decides for itself when to stop: a small head predicts a stop
probability per step. It can stop early and clip the sentence, or never stop and babble until the
length cap. Neither is reproducible on demand from a healthy checkpoint — but the failure below is.

In [ ]:
broken = "In 1984 he paid 25"
ids_b = tokenizer(broken, add_special_tokens=False).input_ids
print(repr(broken), "->", ids_b.count(unk_id), "<unk> of", len(ids_b), "tokens")

unk_wav = np.asarray(speak(broken, speaker, vocoder=vocoder))
print(f"and it still generates {len(unk_wav) / SAMPLING_RATE:.2f}s of confident-sounding audio.")
print("Nothing raised. Nothing warned. That is the failure mode section 2 is about.")
ipd.Audio(unk_wav, rate=SAMPLING_RATE)

## 8 — Evaluating for real

The course's evaluation page recommends **no automatic metric at all**: TTS is one-to-many, so
quality is subjective and MOS — humans scoring 1 to 5 — is the only honest measure. True, and not
very actionable. Two cheap proxies that do say something real:

In [ ]:
from transformers import pipeline
import jiwer

SENTENCES = [
    "the sun provides energy for life on earth",
    "she took the early train from the station",
    "please check my account balance today",
    "a cheaper way to travel is by coach",
]

asr = pipeline("automatic-speech-recognition", model=ASR_ID, device=-1)
refs, hyps = [], []
for text in SENTENCES:
    w = np.asarray(speak(text, speaker, vocoder=vocoder))
    out = asr({"array": w, "sampling_rate": SAMPLING_RATE},
              generate_kwargs={"task": "transcribe", "language": "english", "num_beams": 1})
    refs.append(text)
    hyps.append(out["text"].strip().lower().rstrip(".").strip())
    print(("ok  " if hyps[-1] == text else "DIFF"), hyps[-1])

print("\nround-trip WER:", round(jiwer.wer(refs, hyps), 4))

**A low WER means intelligible, not natural.** It cannot see prosody, pacing, or whether the voice
sounds human — only whether an ASR model recovered the words. A perfect 0.0 is itself the lesson:
these are short, common, in-vocabulary sentences, so the metric has almost no room to discriminate.
Treat it as a regression alarm, not a quality score.

In [ ]:
import librosa

# Did the x-vector change the VOICE, or only the x-vector? Measure the OUTPUT.
pitches = {v: float(np.median(librosa.yin(voice_waves[v], fmin=60, fmax=400, sr=SAMPLING_RATE)))
           for v in VOICES}
for v in VOICES:
    print(f"cmu_us_{v:<4} median f0: {pitches[v]:6.1f} Hz   "
          f"({'female' if v in ('slt', 'clb') else 'male'} in CMU ARCTIC)")

female = np.mean([pitches[v] for v in ("slt", "clb")])
male = np.mean([pitches[v] for v in ("bdl", "rms", "ksp")])
print(f"\nfemale mean {female:.1f} Hz vs male mean {male:.1f} Hz -> separation {female - male:+.1f} Hz")

plt.figure(figsize=(7, 4))
plt.bar(VOICES, [pitches[v] for v in VOICES],
        color=["tab:pink" if v in ("slt", "clb") else "tab:blue" for v in VOICES])
plt.ylabel("median f0 (Hz)"); plt.title("The x-vector reached the output")
plt.tight_layout(); plt.show()

Comparing the input x-vectors to each other would tell you nothing about whether the model *used*
them. Measuring the generated audio does.

For **(c)**, the only measure the course endorses: play the five clips from section 3 without looking
at the labels, score each 1–5 for naturalness, and only then check which voice was which. That is a
MOS panel of one.

## Fine-tuning

The training run lives in `finetune.py` (CPU smoke test by default, `UNIT6_MODE=full` for a GPU) and
the graded hands-on in `colab_handson.ipynb`. The cell below is the recipe in miniature, gated off so
*Run All* stays fast — a real run wants a GPU.

Four settings here are not style choices. Leave any of them out and the run breaks, usually quietly.

In [ ]:
RUN_TRAINING = False   # set True on a GPU box; on CPU this takes hours

In [ ]:
if RUN_TRAINING:
    from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

    args = Seq2SeqTrainingArguments(
        output_dir="speecht5_finetuned_english_dialects",
        per_device_train_batch_size=16, gradient_accumulation_steps=2,
        learning_rate=1e-5, warmup_steps=100, max_steps=1000,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},  # or the first backward raises
        fp16=True, eval_strategy="steps", save_strategy="steps",
        eval_steps=250, save_steps=250, logging_steps=25,
        label_names=["labels"],        # or eval loss is silently never computed
        remove_unused_columns=False,   # or speaker_embeddings is pruned before the collator
        push_to_hub=False, report_to=["none"],
        # predict_with_generate is deliberately absent: generate() returns a spectrogram
    )
    print("see finetune.py for the full recipe, including the collator and the push")
else:
    print("skipped (set RUN_TRAINING = True on a GPU box, or use colab_handson.ipynb)")

## That's Unit 6

The one thing to carry forward: **SpeechT5's vocabulary is 81 characters, and anything outside it
becomes `<unk>` silently.** Audit by tokenizing, never by comparing against `get_vocab()` keys.

And one that costs people the hands-on: `trainer.push_to_hub(tasks="text-to-speech")` writes no
`pipeline_tag`, so the Hub infers `text-to-audio` and the grader cannot see your model. Fix it with
`metadata_update(repo, {"pipeline_tag": "text-to-speech"})`.

**Supplemental reading from the course**

- [HiFi-GAN vocoder](https://arxiv.org/pdf/2010.05646.pdf)
- [X-Vectors](https://www.danielpovey.com/files/2018_icassp_xvectors.pdf)
- [FastSpeech 2](https://arxiv.org/pdf/2006.04558.pdf) — the non-autoregressive alternative
- [MQTTS](https://arxiv.org/pdf/2302.04215v1.pdf) — quantized discrete representations instead of mels